In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
import urllib.request
from umap import UMAP

from src.cf_recon import reconstruct_cf

In [ ]:
# Download Schiebinger et al. 2019 data from the WOT project (Google Drive)
# ~1.4 GB — only runs once; subsequent runs skip if files exist.

data_dir = Path('../data/schiebinger')

In [ ]:
# Load expression matrix
# The WOT ExprMatrix.h5ad is already size-normalised and log-transformed.
adata = sc.read_h5ad(data_dir / 'ExprMatrix.h5ad')

# Merge day labels from cell_days.txt if not already in obs
if 'day' not in adata.obs.columns:
    day_df = pd.read_csv(data_dir / 'cell_days.txt', sep='\t', index_col=0)
    adata.obs = adata.obs.join(day_df)

print(adata)
print('\nobs columns:', adata.obs.columns.tolist())
print('\nSample cell names:')
print(adata.obs_names[:10].tolist())

print('\nDay value counts:')
print(adata.obs['day'].value_counts().sort_index())

In [ ]:
# ── Parse condition from cell names ──────────────────────────────────────────
# Cell names encode the condition (e.g. "serum_..." or "2i_...").
# Extract it and store as a proper obs column.

def _parse_cond(name):
    n = name.lower()
    if 'serum' in n:
        return 'serum'
    elif '2i' in n:
        return '2i'
    else:
        return 'unknown'

adata.obs['condition'] = [_parse_cond(n) for n in adata.obs_names]

COND_COL  = 'condition'
CTRL_VAL  = 'serum'    # control: standard reprogramming medium
TREAT_VAL = '2i'       # treated: 2i medium (higher iPSC yield)

print('Cells per condition:')
print(adata.obs[COND_COL].value_counts())

common_days_all = sorted(
    set(adata.obs.loc[adata.obs[COND_COL] == CTRL_VAL,  'day'].unique()) &
    set(adata.obs.loc[adata.obs[COND_COL] == TREAT_VAL, 'day'].unique())
)
print(f'\nDays present in both conditions ({len(common_days_all)}):')
print(common_days_all)

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
# Data is already normalised + log-transformed by WOT — skip those steps.
adata_pp = adata.copy()

sc.pp.filter_genes(adata_pp, min_cells=50)
print(f'Genes after low-expression filter: {adata_pp.n_vars:,}')

sc.pp.highly_variable_genes(adata_pp, n_top_genes=2000, flavor='seurat')
print(f'Highly variable genes: {adata_pp.var["highly_variable"].sum()}')

sc.tl.pca(adata_pp, use_highly_variable=True, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata_pp, n_neighbors=30, n_pcs=30)
sc.tl.umap(adata_pp, random_state=42)

print('\nDone.')
print(adata_pp)

In [ ]:
# UMAP overview coloured by day and condition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc.pl.umap(adata_pp, color='day',    ax=axes[0], show=False, title='Day',      palette='plasma', frameon=False)
sc.pl.umap(adata_pp, color=COND_COL, ax=axes[1], show=False, title='Condition', frameon=False)

plt.suptitle('Schiebinger et al. 2019 — iPSC reprogramming (Serum vs 2i)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Subsample time points for PT tractability ─────────────────────────────────
# 39 time points → O(38²) OT solves.  Thin to every 2 days instead.
# Also cap cells per (condition, day) at N_MAX to keep OT fast.

DAY_STEP = 1    # keep every DAY_STEP-th integer day
N_MAX    = np.inf  # max cells per (condition, day) bucket

selected_days = sorted([d for d in common_days_all if float(d) == int(float(d))
                         and int(float(d)) % DAY_STEP == 0])
print(f'Selected {len(selected_days)} time points: {selected_days}')

pca_raw = adata_pp.obsm['X_pca'][:, :15]
pca_std = pca_raw.std(axis=0)
pca     = pca_raw / pca_std          # standardise → unit-variance PCs
obs     = adata_pp.obs.reset_index(drop=True)

rng = np.random.default_rng(42)

def get_pca_sub(cond_val, day):
    mask = (obs[COND_COL] == cond_val) & (obs['day'] == day)
    idx  = np.where(mask.values)[0]
    if len(idx) > N_MAX:
        idx = rng.choice(idx, size=N_MAX, replace=False)
    return pca[idx]

ctrl_samples  = [get_pca_sub(CTRL_VAL,  d) for d in selected_days]
treat_samples = [get_pca_sub(TREAT_VAL, d) for d in selected_days]
cf_0          = get_pca_sub(TREAT_VAL, selected_days[0])   # 2i cells at day 0

print('\nCells per step:')
for d, cs, ts in zip(selected_days, ctrl_samples, treat_samples):
    print(f'  day {d:>4}: serum={len(cs)}, 2i={len(ts)}')

In [ ]:
# ── Reconstruct counterfactual ────────────────────────────────────────────────
# CF = 2i cells at day 0, propagated forward under serum (control) velocity fields.
# Question: "What would 2i cells look like if they had reprogrammed in serum medium?"

n_steps = len(selected_days) - 1
print(f'Running reconstruct_cf  (n={n_steps} steps, 15 PCs) ...')
cf_curve = reconstruct_cf(ctrl_samples, cf_0, n=n_steps, project=False)
print('Done.')

In [ ]:
# ── PC1-PC2 trajectory visualisation ─────────────────────────────────────────
COL_CTRL  = '#2E86AB'   # blue   – serum (control)
COL_TREAT = '#D62828'   # red    – 2i (observed)
COL_CF    = '#2DC653'   # green  – counterfactual

ctrl_means  = np.array([s.mean(0) for s in ctrl_samples])
treat_means = np.array([s.mean(0) for s in treat_samples])
cf_means    = np.array([m.locs.mean(0) for m in cf_curve])

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
for ax in axes:
    ax.set_xlabel('PC 1 (standardized)', fontsize=12)
    ax.set_ylabel('PC 2 (standardized)', fontsize=12)

# Left: serum vs 2i actual trajectories
for i in range(len(selected_days)):
    axes[0].scatter(ctrl_samples[i][:, 0],  ctrl_samples[i][:, 1],
                    color=COL_CTRL,  alpha=0.20, s=5, rasterized=True)
    axes[0].scatter(treat_samples[i][:, 0], treat_samples[i][:, 1],
                    color=COL_TREAT, alpha=0.20, s=5, rasterized=True)
axes[0].plot(ctrl_means[:, 0],  ctrl_means[:, 1],  'o-', color=COL_CTRL,  lw=2, ms=8, label='Serum (control)')
axes[0].plot(treat_means[:, 0], treat_means[:, 1], 's-', color=COL_TREAT, lw=2, ms=8, label='2i (observed)')
for i, d in enumerate(selected_days):
    axes[0].annotate(f'd{d}', ctrl_means[i, :2],  xytext=(-12, 4), textcoords='offset points', fontsize=7, color=COL_CTRL)
    axes[0].annotate(f'd{d}', treat_means[i, :2], xytext=(4,   4), textcoords='offset points', fontsize=7, color=COL_TREAT)
axes[0].set_title('Actual trajectories: Serum vs 2i', fontsize=13)
axes[0].legend(fontsize=11)

# Right: actual (faded) + reconstructed CF
for i in range(len(selected_days)):
    axes[1].scatter(ctrl_samples[i][:, 0],  ctrl_samples[i][:, 1],
                    color=COL_CTRL,  alpha=0.08, s=5, rasterized=True)
    axes[1].scatter(treat_samples[i][:, 0], treat_samples[i][:, 1],
                    color=COL_TREAT, alpha=0.08, s=5, rasterized=True)
    axes[1].scatter(cf_curve[i].locs[:, 0], cf_curve[i].locs[:, 1],
                    color=COL_CF,    alpha=0.25, s=5, rasterized=True)
axes[1].plot(ctrl_means[:, 0],  ctrl_means[:, 1],  'o-', color=COL_CTRL,  lw=2, ms=8, alpha=0.4, label='Serum (control)')
axes[1].plot(treat_means[:, 0], treat_means[:, 1], 's-', color=COL_TREAT, lw=2, ms=8, alpha=0.4, label='2i (observed)')
axes[1].plot(cf_means[:, 0],    cf_means[:, 1],    '^-', color=COL_CF,    lw=2, ms=8,            label='CF  (2i → serum dynamics)')
for i, d in enumerate(selected_days):
    axes[1].annotate(f'd{d}', cf_means[i, :2], xytext=(4, 4), textcoords='offset points', fontsize=7, color=COL_CF)
axes[1].set_title('Reconstructed CF: 2i cells under serum dynamics', fontsize=13)
axes[1].legend(fontsize=11)

plt.suptitle('Counterfactual reconstruction — Schiebinger et al. 2019\n'
             'CF starts from 2i at day 0, propagated with serum velocity fields',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Joint UMAP of all three trajectories ─────────────────────────────────────
all_locs   = []
traj_label = []
tp_label   = []

for i, d in enumerate(selected_days):
    for locs, name in [
        (ctrl_samples[i],    'Serum'),
        (treat_samples[i],   '2i'),
        (cf_curve[i].locs,   'CF'),
    ]:
        all_locs.append(locs)
        traj_label.extend([name] * len(locs))
        tp_label.extend([d] * len(locs))

all_locs   = np.vstack(all_locs)
traj_label = np.array(traj_label)
tp_label   = np.array(tp_label, dtype=float)

print(f'Total points for joint UMAP: {len(all_locs):,}')

reducer   = UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=42)
embedding = reducer.fit_transform(all_locs)
u1, u2    = embedding[:, 0], embedding[:, 1]
print('UMAP done.')

# ── Plot ──────────────────────────────────────────────────────────────────────
tp_vals   = np.array(selected_days, dtype=float)
tp_norm   = (tp_vals - tp_vals.min()) / (tp_vals.max() - tp_vals.min())
tp_colors = {d: plt.cm.plasma(v) for d, v in zip(selected_days, tp_norm)}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

for ax, (hi, title) in zip(axes, [('Serum', 'Serum (control)'),
                                   ('2i',   '2i (observed)'),
                                   ('CF',   'CF (2i → serum dynamics)')]):
    gray_mask = traj_label != hi
    ax.scatter(u1[gray_mask], u2[gray_mask],
               c='#cccccc', s=3, alpha=0.15, linewidths=0, rasterized=True)

    for d in selected_days:
        mask = (traj_label == hi) & (tp_label == d)
        ax.scatter(u1[mask], u2[mask],
                   c=[tp_colors[d]], s=6, alpha=0.6, linewidths=0,
                   rasterized=True, label=f'd{int(d)}')

    ax.set_title(title, fontsize=13)
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.axis('off')

axes[0].set_ylabel('UMAP 2', fontsize=11)
axes[-1].legend(title='Day', fontsize=8, title_fontsize=9,
                loc='best', markerscale=2, ncol=2)

plt.suptitle('Joint UMAP — Schiebinger et al. 2019\n'
             '(color = day, gray = other trajectories)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()